# Reasoning & Logic Techniques — Interactive Comparison

This notebook demonstrates Chain-of-Thought (CoT), Self-Consistency, and Plan-and-Solve prompting
by comparing their outputs on identical problems. Run top-to-bottom to see the differences.

In [ ]:
# Configuration — set your provider and model here
PROVIDER = "openai"  # "openai", "anthropic", or "ollama"
MODEL = None          # None = provider default, or specify e.g. "gpt-4o", "claude-sonnet-4-20250514"

# Self-consistency settings
SC_SAMPLES = 5       # Number of samples for self-consistency
SC_TEMPERATURE = 0.7 # Temperature for diverse sampling

In [ ]:
import sys
sys.path.insert(0, "..")

from utils.llm_client import call_llm
from collections import Counter
import re
import time

---
## 1. Direct Prompting vs. Chain-of-Thought

The same math word problem, solved two ways: one without reasoning instructions, one with.

In [ ]:
problem = (
    "A juggler has 12 balls. He loses 3 and buys 5 times as many as he lost. "
    "How many balls does he have now?"
)

# Direct prompting — no reasoning instruction
direct_prompt = f"Q: {problem}\nA:"

# Chain-of-Thought — with reasoning trigger
cot_prompt = f"Q: {problem}\nA: Let's think step by step."

print("=" * 60)
print("DIRECT PROMPTING")
print("=" * 60)
direct_response = call_llm(direct_prompt, provider=PROVIDER, model=MODEL, temperature=0.0)
print(direct_response)

print()
print("=" * 60)
print("CHAIN-OF-THOUGHT")
print("=" * 60)
cot_response = call_llm(cot_prompt, provider=PROVIDER, model=MODEL, temperature=0.0)
print(cot_response)

### Analysis

The direct answer may or may not be correct, but you cannot tell *where* the model went wrong.
The CoT response shows the reasoning chain — if the answer is wrong, you can inspect each step
to find the error. **Diagnosability is the primary engineering value of CoT.**

---
## 2. Self-Consistency: Multiple Paths, Majority Vote

Run the same CoT prompt multiple times at a higher temperature, extract answers, and vote.

In [ ]:
def extract_final_number(response: str) -> str:
    """Extract the last number mentioned in a response as the final answer."""
    # Look for patterns like 'the answer is X' or 'answer: X'
    patterns = [
        r"the answer is\s*\$?([\d,\.]+)",
        r"answer:\s*\$?([\d,\.]+)",
        r"=\s*\$?([\d,\.]+)\s*$",
    ]
    for pattern in patterns:
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            return match.group(1).replace(",", "")
    
    # Fallback: find all numbers and return the last one
    numbers = re.findall(r"\$?([\d,]+\.?\d*)", response)
    if numbers:
        return numbers[-1].replace(",", "")
    return response.strip()

def self_consistency(prompt_template: str, problem: str, n_samples: int = 5,
                     temperature: float = 0.7) -> dict:
    """Run self-consistency voting on a problem."""
    full_prompt = prompt_template.format(problem=problem)
    answers = []
    
    for i in range(n_samples):
        response = call_llm(
            full_prompt, provider=PROVIDER, model=MODEL,
            temperature=temperature
        )
        answer = extract_final_number(response)
        answers.append(answer)
    
    # Majority vote
    counts = Counter(answers)
    most_common_answer, most_common_count = counts.most_common(1)[0]
    confidence = most_common_count / n_samples
    
    return {
        "all_answers": answers,
        "vote_counts": dict(counts),
        "final_answer": most_common_answer,
        "confidence": confidence,
        "consistent": confidence >= 0.6,
    }

In [ ]:
sc_problem = (
    "Janet's ducks lay 16 eggs per day. She eats three for breakfast "
    "every morning and bakes muffins for her friends every day with four. "
    "She sells the remainder for $2 per egg. How much does she make every day?"
)

sc_prompt_template = "Q: {problem}\nA: Let's think step by step."

print(f"Running Self-Consistency with {SC_SAMPLES} samples...")
print()

start = time.time()
result = self_consistency(sc_prompt_template, sc_problem, SC_SAMPLES, SC_TEMPERATURE)
elapsed = time.time() - start

print("Individual answers:")
for i, ans in enumerate(result["all_answers"], 1):
    print(f"  Sample {i}: {ans}")

print()
print(f"Vote counts: {result['vote_counts']}")
print(f"Final answer: {result['final_answer']}")
print(f"Confidence: {result['confidence']:.0%}")
print(f"Consistent: {result['consistent']}")
print(f"Elapsed: {elapsed:.1f}s")

### Analysis

Self-consistency gives you a confidence signal: if 4 out of 5 samples agree on $18, you can trust
the answer more than a single sample. If the vote is split (e.g., 2 say $18, 2 say $14, 1 says $20),
the model is uncertain and the answer should be escalated or verified.

---
## 3. Plan-and-Solve: Structured Decomposition

Instead of "think step by step", we explicitly ask the model to plan, then execute.

In [ ]:
ps_problem = (
    "Grace weighs 125 pounds. Alex weighs 498 pounds. "
    "Together with their dog Rex, the three weigh 700 pounds. "
    "How much does Rex weigh?"
)

# Zero-shot CoT
cot_prompt = f"Q: {ps_problem}\nA: Let's think step by step."

# Plan-and-Solve
ps_prompt = (
    f"Q: {ps_problem}\n"
    f"A: Let's first understand the problem and devise a plan to solve the problem. "
    f"Then, let's carry out the plan and solve the problem step by step."
)

# PS+ (enhanced)
ps_plus_prompt = (
    f"Q: {ps_problem}\n"
    f"A: Let's first understand the problem, extract relevant variables and their "
    f"corresponding numerals, and devise a plan to solve the problem. "
    f"Then, let's carry out the plan, calculate intermediate results, and solve "
    f"the problem step by step. Pay attention to calculation and commonsense."
)

print("=" * 60)
print("ZERO-SHOT CHAIN-OF-THOUGHT")
print("=" * 60)
cot_resp = call_llm(cot_prompt, provider=PROVIDER, model=MODEL, temperature=0.0)
print(cot_resp)

print()
print("=" * 60)
print("PLAN-AND-SOLVE")
print("=" * 60)
ps_resp = call_llm(ps_prompt, provider=PROVIDER, model=MODEL, temperature=0.0)
print(ps_resp)

print()
print("=" * 60)
print("PLAN-AND-SOLVE+ (ENHANCED)")
print("=" * 60)
ps_plus_resp = call_llm(ps_plus_prompt, provider=PROVIDER, model=MODEL, temperature=0.0)
print(ps_plus_resp)

### Analysis

The Plan-and-Solve versions explicitly decompose the problem into identified variables and subtasks.
PS+ adds explicit instructions to extract variables and pay attention to calculations, which
reduces missing-step errors on complex problems.

---
## Summary

| Technique | When to use | Cost | Accuracy signal |
|---|---|---|---|
| Direct prompting | Simple extraction, classification | 1x | None — black box |
| Zero-shot CoT | Multi-step reasoning on non-reasoning models | 1x | Reasoning chain (diagnosability) |
| Self-Consistency | Accuracy-critical tasks with canonical answers | Nx (3-5x) | Confidence from agreement |
| Plan-and-Solve | Complex multi-variable problems | 1x | Structured decomposition |

**Key insight**: The right technique depends on your model class, task type, and accuracy/cost requirements.
Always benchmark against a direct baseline before adopting any reasoning technique.